# Detección de Anomalías en Tráfico de Red
**Estudiante:** Luis Alberto Quilla Lopez
**Dataset:** network_traffic.csv (10000 registros)
**Modelo:** Isolation Forest

## 3.1 Exploración y Preprocesamiento

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
import joblib
import os

os.chdir(os.path.expanduser('~/examen-practico-quilla-lopez'))
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12,5)

df = pd.read_csv('lab3/network_traffic.csv')
print('Shape:', df.shape)
df.head()

In [ ]:
print('Estadísticas descriptivas:')
df.describe()

In [ ]:
print('Valores nulos:')
print(df.isnull().sum())
print()
print('Distribución de etiquetas:')
print(df['label'].value_counts())

In [ ]:
fig, axes = plt.subplots(1,2)
axes[0].hist(df['bytes_sent'], bins=50, color='steelblue', edgecolor='black')
axes[0].set_title('Distribución de bytes_sent')
axes[0].set_xlabel('Bytes enviados')
axes[1].hist(df['duration_sec'], bins=50, color='coral', edgecolor='black')
axes[1].set_title('Distribución de duration_sec')
axes[1].set_xlabel('Duración (segundos)')
plt.tight_layout()
plt.savefig('lab3/evidencias/SCR-3.1_eda.png', dpi=150)
plt.show()
print('[OK] Histogramas generados')

In [ ]:
# Tratar valores atípicos extremos (percentil 99)
for col in ['bytes_sent','bytes_recv','duration_sec','packets']:
    q99 = df[col].quantile(0.99)
    df[col] = df[col].clip(upper=q99)
    print(f'{col}: limitado a percentil 99 ({q99:.2f})')

In [ ]:
# Feature engineering
df['ratio_bytes'] = df['bytes_sent'] / (df['bytes_recv'] + 1)
df['bytes_por_segundo'] = (df['bytes_sent'] + df['bytes_recv']) / (df['duration_sec'] + 0.001)
df['total_bytes'] = df['bytes_sent'] + df['bytes_recv']
print('Nuevas features creadas: ratio_bytes, bytes_por_segundo, total_bytes')
df[['ratio_bytes','bytes_por_segundo','total_bytes']].head()

In [ ]:
# Normalizar features numéricas
feature_cols = ['bytes_sent','bytes_recv','duration_sec','packets','ratio_bytes','bytes_por_segundo','total_bytes']
scaler = StandardScaler()
df_scaled = scaler.fit_transform(df[feature_cols])
print('Features normalizadas con StandardScaler')
print('Shape:', df_scaled.shape)

## 3.2 Entrenamiento del Modelo

In [ ]:
# Isolation Forest
model = IsolationForest(
    contamination=0.05,
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
model.fit(df_scaled)
df['pred'] = model.predict(df_scaled)
# Isolation Forest: -1 = anomalía, 1 = normal
df['pred_label'] = df['pred'].map({1: 'normal', -1: 'anomaly'})
print('Predicciones completadas')
print(df['pred_label'].value_counts())

In [ ]:
# Métricas de evaluación
y_true = df['label'].map({'normal': 1, 'anomaly': -1})
y_pred = df['pred']

precision = precision_score(y_true, y_pred, pos_label=-1)
recall = recall_score(y_true, y_pred, pos_label=-1)
f1 = f1_score(y_true, y_pred, pos_label=-1)

print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1-Score: {f1:.4f}')

cm = confusion_matrix(y_true, y_pred, labels=[1, -1])
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal','Anomalía'],
            yticklabels=['Normal','Anomalía'])
plt.title('Matriz de Confusión')
plt.xlabel('Predicción')
plt.ylabel('Real')
plt.tight_layout()
plt.savefig('lab3/evidencias/SCR-3.2_metricas.png', dpi=150)
plt.show()

## 3.3 Interpretación y Umbral Dinámico

In [ ]:
# Obtener scores de anomalía
scores = model.decision_function(df_scaled)
df['anomaly_score'] = scores

plt.figure()
plt.hist(scores, bins=50, color='purple', edgecolor='black', alpha=0.7)
plt.axvline(x=0, color='red', linestyle='--', label='Umbral default (score=0)')
plt.xlabel('Anomaly Score')
plt.ylabel('Frecuencia')
plt.title('Distribución de Scores de Anomalía')
plt.legend()
plt.tight_layout()
plt.savefig('lab3/evidencias/SCR-3.3_umbral_f1.png', dpi=150)
plt.show()

In [ ]:
# Curva umbral vs F1-Score
thresholds = np.linspace(scores.min(), scores.max(), 100)
f1_scores = []
for thr in thresholds:
    y_pred_thr = np.where(scores < thr, -1, 1)
    f1_scores.append(f1_score(y_true, y_pred_thr, pos_label=-1))

best_idx = np.argmax(f1_scores)
best_thr = thresholds[best_idx]
best_f1 = f1_scores[best_idx]

plt.figure()
plt.plot(thresholds, f1_scores, color='green')
plt.axvline(x=best_thr, color='red', linestyle='--', label=f'Mejor umbral: {best_thr:.4f} (F1={best_f1:.4f})')
plt.xlabel('Umbral')
plt.ylabel('F1-Score')
plt.title('Curva Umbral vs F1-Score')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('lab3/evidencias/SCR-3.3_umbral_f1.png', dpi=150)
plt.show()
print(f'Mejor umbral: {best_thr:.4f} con F1-Score: {best_f1:.4f}')

In [ ]:
# Top 10 registros más anómalos
df['anomaly_score'] = scores
top_anomalies = df.nsmallest(10, 'anomaly_score')
print('=== TOP 10 REGISTROS MÁS ANÓMALOS ===')
cols_show = ['timestamp','src_ip','dst_ip','dst_port','protocol','bytes_sent','bytes_recv','duration_sec','packets','label','anomaly_score']
top_anomalies[cols_show].reset_index(drop=True)

**Explicación de amenazas reales:**
1. **Puertos no estándar**: Varias conexiones a puertos poco comunes (21=FTP, 25=SMTP, etc.) pueden ser intentos de exfiltración.
2. **Volumen alto de datos**: Registros con bytes_sent o bytes_recv extremadamente altos pueden indicar transferencia masiva de datos no autorizada.
3. **Protocolos inusuales**: ICMP con alto volumen puede ser un túnel de datos encubierto.
4. **IPs externas sospechosas**: Direcciones IP externas con comportamiento anómalo en horarios no laborales.
5. **Duración de conexión**: Conexiones muy largas o muy cortas con alto volumen sugieren actividad maliciosa.

## 3.4 Exportación del Modelo

In [ ]:
# Guardar modelo y scaler
joblib.dump(model, 'lab3/modelo_anomalias.pkl')
joblib.dump(scaler, 'lab3/scaler.pkl')
joblib.dump(feature_cols, 'lab3/feature_cols.pkl')
print('[OK] Modelo y preprocesador exportados')

# Verificar
print('Archivos generados:')
import os
for f in ['modelo_anomalias.pkl','scaler.pkl','feature_cols.pkl']:
    path = f'lab3/{f}'
    print(f'  {path}: {os.path.getsize(path)} bytes')